# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR\^2](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema, which describes its structure and semantics. Source schema:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install --quiet mlcroissant

## 1. Data Loading

We will load the dataset's Croissant metadata and initialize a `mlcroissant.Dataset` for further inspection and extraction. This also displays dataset summary information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata
meta = dataset.metadata
print(f"\u001b[1mDataset Name:\u001b[0m {meta.name}\n")
print(f"\u001b[1mDescription:\u001b[0m {meta.description}\n")
print(f"\u001b[1mCitation:\u001b[0m {getattr(meta, 'citeAs', '')}")


## 2. Data Overview

Let's enumerate all record sets (tables) in the dataset, along with their `@id`s and their available fields (columns) and their `@id`s. Referencing by `@id` ensures consistent and precise access to dataset elements.


In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets.values())
for rs in record_sets:
    print(f"\n\u001b[1mRecordSet Name:\u001b[0m {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields:")
    for fld in rs.fields:
        print(f"    - {fld.name} (@id: {fld.id}) (dataType: {fld.data_type})")

Let's quickly display a few records for each record set, using the `@id` value.

In [ ]:
# Preview records for each record set by @id
for rs in record_sets:
    print(f"\nDisplaying sample records for RecordSet: {rs.name} (@id: {rs.id})")
    try:
        rows = list(dataset.records(record_set=rs.id))
        for i, rec in enumerate(rows[:3]):
            print(f"  Record {i+1}: {rec}")
        if not rows:
            print("  No records found.")
    except Exception as e:
        print(f"  Error loading records: {e}")

## 3. Data Extraction

Let's extract the main tabular dataset as a DataFrame for further analysis.

**Tip:** Use the record set(s) `@id` you observed above. For this dataset, there is likely a primary record set holding patient/clinical variables. We'll extract all available record sets, but focus on the main one for EDA.


In [ ]:
# Extract all available record sets into DataFrames by @id

df_dict = {}
for rs in record_sets:
    try:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        df_dict[rs.id] = df
        print(f"Loaded RecordSet '{rs.name}' (@id: {rs.id}), shape: {df.shape}")
    except Exception as e:
        print(f"Could not load records for RecordSet '{rs.name}': {e}")

Next, let's inspect columns for the main (first) record set, and display the first few records.

In [ ]:
# Choose the first (main) record set for EDA
main_rs = record_sets[0]  # If multiple relevant, adjust as needed
main_rs_id = main_rs.id
main_df = df_dict[main_rs_id]

print(f"\u001b[1mColumns in RecordSet '{main_rs.name}' (@id: {main_rs_id}):\u001b[0m")
pprint.pprint(list(main_df.columns))
main_df.head()

## 4. Exploratory Data Analysis (EDA)

We'll perform basic EDA steps:

1. **Filter:** Select records where a numeric field exceeds a threshold.
2. **Normalize:** Standardize a numeric field.
3. **Group:** Aggregate by a grouping field.

First, let's identify numeric and categorical fields via metadata.

In [ ]:
# Identify a numeric field and a grouping field by @id
numeric_fields = [f for f in main_rs.fields if f.data_type.lower() in ["number", "float", "integer"]]
categorical_fields = [f for f in main_rs.fields if f.data_type.lower() in ["text", "string"]]

if not numeric_fields:
    raise Exception("No numeric fields found in this record set.")

numeric_field = numeric_fields[0]
numeric_field_id = numeric_field.id
numeric_field_name = numeric_field.name
print(f"Using numeric field: {numeric_field_name} (@id: {numeric_field_id})")

# Pick a grouping (categorical) field that is not the index, or adjust if necessary
group_field = None
for f in categorical_fields:
    if f.name.lower() not in ["id", "index"]:
        group_field = f
        break

if not group_field:
    raise Exception("No suitable grouping field found.")
group_field_id = group_field.id
group_field_name = group_field.name
print(f"Using grouping field: {group_field_name} (@id: {group_field_id})")


In [ ]:
# EDA: Filter, normalize, group

col_numeric = numeric_field_id
col_group = group_field_id

if col_numeric in main_df.columns:
    s = pd.to_numeric(main_df[col_numeric], errors='coerce')
    thresh = s.mean() if pd.notnull(s.mean()) else 10
    filtered_df = main_df.loc[s > thresh].copy()
    print(f"Filtered records with {col_numeric} > {thresh:.2f}:")
    print(filtered_df[[col_numeric, col_group]].head())

    norm_col = f"{col_numeric}_normalized"
    filtered_df[norm_col] = (pd.to_numeric(filtered_df[col_numeric], errors='coerce') - s.mean()) / s.std()
    print(f"\nNormalized {col_numeric} for filtered records:")
    print(filtered_df[[col_numeric, norm_col]].head())

    # Grouping and aggregation
    if col_group in filtered_df.columns and filtered_df[col_group].nunique() > 1:
        grouped = filtered_df.groupby(col_group)[col_numeric].mean()
        print(f"\nMean {col_numeric} grouped by {col_group}:")
        print(grouped.head())
else:
    print(f"Column {col_numeric} not found in DataFrame columns: {main_df.columns.tolist()}")

## 5. Visualization

Let's plot the distribution of the selected numeric field, and show the mean by each group.

In [ ]:
# Visualization of numeric field and group means
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Histogram
plt.figure(figsize=(7, 4))
s_all = pd.to_numeric(main_df[col_numeric], errors='coerce')
sns.histplot(s_all.dropna(), kde=True, bins=15)
plt.title(f"Distribution of {col_numeric}")
plt.xlabel(col_numeric)
plt.ylabel("Count")
plt.show()

# If grouping
if col_group in main_df.columns and main_df[col_group].nunique() <= 20:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=main_df[col_group], y=s_all)
    plt.title(f"{col_numeric} by {col_group}")
    plt.xlabel(col_group)
    plt.ylabel(col_numeric)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

Using `mlcroissant`, we explored the FAIR^2 colorectal cancer dataset using robust metadata-driven access with `@id` references. We extracted records, performed filtering, normalization, grouping, and visualization. This approach ensures transparent, reproducible, and metadata-guided research workflows on FAIR datasets.

You may now proceed with more domain-specific or advanced analyses, using field and record set `@id`s throughout your workflow for best practice.